# Invest-a-Bull ? Live Top 5 Trending Stocks Notebook

This notebook replaces the old prompt-driven workflow with a reproducible, package-based analysis pipeline.

Run the cells from top to bottom to:

1. pull the latest trending top 5 U.S. stocks
2. refresh the project artifacts
3. review portfolio and Monte Carlo outputs
4. compare low/medium/high allocation scenarios


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path(r'C:\Users\big_j\PycharmProjects\Invest-a-Bull')

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from MCForecastTools import MCSimulation
from invest_a_bull.config import AnalysisConfig
from invest_a_bull.pipeline import run_pipeline

plt.style.use('seaborn-v0_8')
pd.options.display.float_format = '{:,.4f}'.format


In [ ]:
capital = 15_000
duration_years = 1
top_n = 5

config = AnalysisConfig(
    top_n=top_n,
    monte_carlo_days=252 * duration_years,
    monte_carlo_simulations=250,
)

display(Markdown(
    f'**Capital assumption:** ${capital:,.0f}  \
**Forecast horizon:** {duration_years} year(s)  \
**Top N selection:** {top_n}'
))


In [ ]:
outputs = run_pipeline(config)
metadata = json.loads(Path(outputs['metadata']).read_text(encoding='utf-8'))
top_selection = pd.read_csv(outputs['top_selection'])
top_prices = pd.read_csv(outputs['top_prices'], index_col=0, parse_dates=True)
portfolio_summary = pd.read_csv(outputs['portfolio_summary'])
mc_summary = pd.read_csv(outputs['monte_carlo_summary'], index_col=0).iloc[:, 0]

artifact_table = pd.DataFrame({
    'Artifact': list(outputs.keys()),
    'Path': [str(Path(path).relative_to(PROJECT_ROOT)) for path in outputs.values()],
})

display(Markdown(
    f"**Data as of:** {metadata['data_as_of']}  \
**Selection source:** {metadata['selection_source']}  \
**Selected tickers:** {', '.join(metadata['selected_tickers'])}"
))
top_selection


In [ ]:
display(Markdown('## Saved artifacts'))
artifact_table


In [ ]:
display(Markdown('## Portfolio summary'))
portfolio_summary


In [ ]:
display(Markdown('## Monte Carlo summary'))
mc_summary.to_frame('value')


In [ ]:
normalized_prices = top_prices / top_prices.iloc[0]
ax = normalized_prices.plot(figsize=(12, 6), linewidth=2)
ax.set_title('Top 5 Trending Stocks ? Normalized Performance')
ax.set_ylabel('Growth of $1')
ax.set_xlabel('Date')
ax.grid(alpha=0.25)
plt.show()


In [ ]:
returns = top_prices.pct_change().dropna()
corr = returns.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0, fmt='.2f')
plt.title('Top 5 Trending Stocks ? Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:
def build_profile_weights(num_assets: int):
    base = np.arange(1, num_assets + 1, dtype=float)
    low = base / base.sum()
    medium = np.repeat(1 / num_assets, num_assets)
    high = low[::-1]
    return {
        'Low Risk': low,
        'Medium Risk': medium,
        'High Risk': high,
    }

volatility_rank = returns.std().sort_values(ascending=False).index.tolist()
ordered_close = top_prices[volatility_rank]
multi_level = pd.concat([ordered_close], axis=1, keys=['close']).swaplevel(0, 1, 1)

profile_rows = []
profile_paths = {}
for profile_name, weights in build_profile_weights(len(volatility_rank)).items():
    simulation = MCSimulation(
        portfolio_data=multi_level,
        weights=weights.tolist(),
        num_simulation=100,
        num_trading_days=252 * duration_years,
    )
    simulation.calc_cumulative_return()
    summary = simulation.summarize_cumulative_return()
    profile_rows.append({
        'Profile': profile_name,
        'Expected Terminal Value': summary['mean'],
        '95% CI Lower': summary['95% CI Lower'],
        '95% CI Upper': summary['95% CI Upper'],
    })
    profile_paths[profile_name] = simulation.simulated_return.mean(axis=1)

profile_summary = pd.DataFrame(profile_rows)
profile_summary


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for profile_name, path in profile_paths.items():
    ax.plot(path.values, label=profile_name, linewidth=2)
ax.set_title('Risk Profiles ? Mean Monte Carlo Path')
ax.set_xlabel('Trading Days')
ax.set_ylabel('Cumulative Value')
ax.legend()
ax.grid(alpha=0.25)
plt.show()


In [33]:
def projected_lending_profit(principal: float, annual_rate: float, years: int) -> float:
    months = max(int(years * 12), 1)
    return principal * ((1 + annual_rate / 12) ** months - 1)

allocations = {
    'Low Risk': {'portfolio_pct': 0.80, 'lending_pct': 0.20, 'lending_rate': 0.075},
    'Medium Risk': {'portfolio_pct': 0.60, 'lending_pct': 0.40, 'lending_rate': 0.10},
    'High Risk': {'portfolio_pct': 0.50, 'lending_pct': 0.50, 'lending_rate': 0.14},
}

scenario_rows = []
for _, row in profile_summary.iterrows():
    settings = allocations[row['Profile']]
    portfolio_capital = capital * settings['portfolio_pct']
    lending_capital = capital * settings['lending_pct']
    lending_profit = projected_lending_profit(lending_capital, settings['lending_rate'], duration_years)

    scenario_rows.append({
        'Profile': row['Profile'],
        'Portfolio Allocation': f'${portfolio_capital:,.2f}',
        'Lending Allocation': f'${lending_capital:,.2f}',
        'Expected Portfolio Value': f'${row['Expected Terminal Value'] * portfolio_capital:,.2f}',
        '95% Portfolio Floor': f'${row['95% CI Lower'] * portfolio_capital:,.2f}',
        '95% Portfolio Ceiling': f'${row['95% CI Upper'] * portfolio_capital:,.2f}',
        'Projected Lending Profit': f'${lending_profit:,.2f}',
    })

scenario_table = pd.DataFrame(scenario_rows)
display(Markdown('## Capital allocation scenarios'))
scenario_table


NameError: name 'profile_summary' is not defined